<a href="https://colab.research.google.com/github/Boulder1-kihara/gemma4-RAG-study-helper/blob/main/study_helper.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
!pip install langchain langchain-community langchain-chroma sentence-transformers
!pip install -qU langchain langchain-community langchain-core langchain-text-splitters chromadb sentence-transformers duckduckgo-search langchain-google-genai pypdf


In [ ]:
# 1. Install the missing zstd extractor
!apt-get install -y zstd

# 2. Download and install Ollama
!curl -fsSL https://ollama.com/install.sh | sh

# 3. Start the local server in the background
!nohup ollama serve > ollama.log 2>&1 &

# 4. Pull the Gemma 4 model
!ollama pull gemma4

In [ ]:
# 1. Install the system OCR tool so the agent can read text inside images
!apt-get install -y tesseract-ocr
!apt-get install -y poppler-utils

# 2. Install unstructured with the "all-docs" flag to handle Word, PPT, etc.
!pip install -qU "unstructured[all-docs]" langchain-community

In [ ]:
!pip install -qU "unstructured[all-docs]" langchain langchain-community langchain-core langchain-text-splitters chromadb sentence-transformers duckduckgo-search pypdf langchain-huggingface langchain-ollama

In [ ]:
!apt-get update
!apt-get install -y libreoffice
!soffice --version

In [ ]:
import os
import shutil
from langchain_community.vectorstores import Chroma

persist_dir = "/content/drive/MyDrive/Deep_Intel_Brain"

# Clear out the directory if a broken version exists
if os.path.exists(persist_dir):
    shutil.rmtree(persist_dir)

print(f"Saving {len(chunks)} chunks permanently to Google Drive...")

# Save it using the chunks and embeddings already in your RAM
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=persist_dir
)
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

print("Success! Your agent's brain is now safely stored on Google Drive.")

In [ ]:
!pip install -U ddgs

from langchain_core.tools import create_retriever_tool
from langchain_community.tools import DuckDuckGoSearchRun

# Tool 1: The Local Notes Search
notes_tool = create_retriever_tool(
    retriever,
    name="university_notes_search",
    description="Search for information, definitions, and concepts from the user's official class notes. Always use this tool first."
)

# Tool 2: The Web Fallback
web_search = DuckDuckGoSearchRun()
web_tool = web_search

tools = [notes_tool, web_tool]

In [ ]:
from langchain_ollama import ChatOllama
from langgraph.prebuilt import create_react_agent

# 1. Initialize the LLM with the CURRENT active model version
llm = ChatOllama(
    model="gemma4",
    temperature=0
)

# 2. Define your system instructions (Notice the spaces added at the end of the lines!)
system_prompt = (
    "You were created by Abel, an engineer at Deep Intel. Your name is 'University Study Helper'. "
    "You are a helpful university tutor. Answer the student's past paper questions. "
    "First, search the 'university_notes_search' tool. If the answer is complete, use it. "
    "If the notes do not contain the answer or are missing details, use the web search tool to find the rest."
)

# 3. Create the LangGraph Agent
agent_executor = create_react_agent(llm, tools, prompt=system_prompt)

In [ ]:
# Wake up the Ollama background server
!nohup ollama serve > ollama.log 2>&1 &

# Give it a second to boot, then verify Gemma 4 is ready
!sleep 3
!ollama list

In [ ]:
from IPython.display import display, Markdown

# The question from your past paper
question = input("Type your past paper question here: ")

print("\nSearching notes and firing up Gemma 4 locally...")

# Invoke the local agent
response = agent_executor.invoke({"messages": [("human", question)]})

# Extract the text and render it beautifully
raw_content = response["messages"][-1].content
if isinstance(raw_content, list):
    final_text = raw_content[0].get("text", "")
else:
    final_text = raw_content

display(Markdown(final_text))